In [ ]:
# ============================================
# FATIGUE DETECTOR: MEDIAPIPE HEAD POSE + OLD EYES/YAWN
# STABLE COUNTS + ALARM ONLY ON SLAPEN
# ============================================

import cv2
import urllib.request
import os
import numpy as np
import torch
import torch.nn as nn
import time
import threading
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import mediapipe as mp

try:
    import winsound
except:
    winsound = None

print("Imports geladen!")

# ============================================
# DEVICE
# ============================================
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print('🚀 MPS!')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print('🚀 CUDA!')
else:
    device = torch.device('cpu')
    print('CPU')

# ============================================
# MODELS
# ============================================
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*28*28, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.model(x)

eyes_model = SimpleCNN().to(device)
eyes_model.load_state_dict(torch.load(
    '/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/Model/eyes/fatigue_model.pth',
    map_location=device
))
eyes_model.eval()
print('✅ Eyes OK')

yawn_model = models.efficientnet_b0(weights='DEFAULT')
yawn_model.classifier[1] = nn.Linear(yawn_model.classifier[1].in_features, 2)
yawn_model.load_state_dict(torch.load(
    '/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/Model/yawn/yawn_model.pth',
    map_location=device
))
yawn_model.to(device)
yawn_model.eval()
print('✅ Yawn OK')

# ============================================
# TRANSFORMS
# ============================================
eyes_transform = transforms.Compose([transforms.ToTensor()])
yawn_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ============================================
# MEDIA PIPE
# ============================================
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

MOUTH_LEFT = 61
MOUTH_RIGHT = 291
MOUTH_TOP = 13
MOUTH_BOTTOM = 14

# Pose landmarks
NOSE_TIP = 1
CHIN = 152
LEFT_EYE_LEFT = 33
RIGHT_EYE_RIGHT = 263
LEFT_MOUTH = 61
RIGHT_MOUTH = 291

# ============================================
# CONFIG
# ============================================
CONF_THRESHOLD = 0.70
YAWN_THRESHOLD = 0.62
CLOSED_ALARM_SECONDS = 2.0
INFERENCE_SKIP_FRAMES = 2
STABLE_FRAMES = 2

# ============================================
# STATE
# ============================================
blink_count = 0
yawn_count = 0
eyes_closed_since = None
was_closed_prev = False
was_yawn_prev = False
alarm_active = False
last_alarm = 0.0

cached_eyes_closed = False
cached_is_yawn = False
cached_eyes_conf = 0.0
cached_yawn_conf = 0.0

stable_eye_state = None
stable_eye_frames = 0
stable_yawn_state = None
stable_yawn_frames = 0

fps = 0.0
fps_n = 0
fps_t0 = time.time()

# ============================================
# AUDIO
# ============================================
def play_loud_alarm():
    if winsound:
        for _ in range(10):
            winsound.Beep(1200, 300)
            time.sleep(0.1)
    else:
        for _ in range(15):
            print("\a")
            time.sleep(0.2)

def start_loud_alarm():
    threading.Thread(target=play_loud_alarm, daemon=True).start()

def clamp(v, lo, hi):
    return max(lo, min(v, hi))

def head_pose_label(pitch, yaw, roll):
    if yaw < -15:
        side = "LEFT"
    elif yaw > 15:
        side = "RIGHT"
    else:
        side = "FORWARD"

    if pitch < -15:
        vert = "DOWN"
    elif pitch > 15:
        vert = "UP"
    else:
        vert = ""

    if vert:
        return f"{side} / {vert}"
    return side

def estimate_head_pose(face_landmarks, w, h):
    image_points = np.array([
        (face_landmarks[NOSE_TIP].x * w, face_landmarks[NOSE_TIP].y * h),
        (face_landmarks[CHIN].x * w, face_landmarks[CHIN].y * h),
        (face_landmarks[LEFT_EYE_LEFT].x * w, face_landmarks[LEFT_EYE_LEFT].y * h),
        (face_landmarks[RIGHT_EYE_RIGHT].x * w, face_landmarks[RIGHT_EYE_RIGHT].y * h),
        (face_landmarks[LEFT_MOUTH].x * w, face_landmarks[LEFT_MOUTH].y * h),
        (face_landmarks[RIGHT_MOUTH].x * w, face_landmarks[RIGHT_MOUTH].y * h),
    ], dtype=np.float64)

    model_points = np.array([
        (0.0, 0.0, 0.0),
        (0.0, -330.0, -65.0),
        (-225.0, 170.0, -135.0),
        (225.0, 170.0, -135.0),
        (-150.0, -150.0, -125.0),
        (150.0, -150.0, -125.0)
    ], dtype=np.float64)

    focal_length = w
    center = (w / 2, h / 2)
    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]
    ], dtype=np.float64)
    dist_coeffs = np.zeros((4, 1))

    success, rotation_vector, translation_vector = cv2.solvePnP(
        model_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE
    )

    if not success:
        return None, None, None

    rmat, _ = cv2.Rodrigues(rotation_vector)
    angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)
    pitch = angles[0]
    yaw = angles[1]
    roll = angles[2]
    return pitch, yaw, roll

# ============================================
# CAMERA
# ============================================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1000)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 750)
cap.set(cv2.CAP_PROP_FPS, 30)

print("🎥 MEDIAPIPE HEAD POSE + OLD MODELS | q=quit")

inference_frame_count = 0

# ============================================
# MAIN LOOP
# ============================================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    fps_n += 1
    if time.time() - fps_t0 >= 1.0:
        fps = fps_n / (time.time() - fps_t0)
        fps_t0 = time.time()
        fps_n = 0

    now = time.time()
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = face_mesh.process(rgb)

    inference_frame_count += 1
    do_inference = (inference_frame_count % INFERENCE_SKIP_FRAMES == 0)

    is_closed_now = cached_eyes_closed
    is_yawn_now = cached_is_yawn
    eyes_conf = cached_eyes_conf
    yawn_conf = cached_yawn_conf

    pose_text = "NO FACE"
    fx = fy = fw = fh = 0
    mx1 = my1 = mx2 = my2 = 0

    if result.multi_face_landmarks and do_inference:
        face_landmarks = result.multi_face_landmarks[0].landmark

        xs = [int(lm.x * w) for lm in face_landmarks]
        ys = [int(lm.y * h) for lm in face_landmarks]

        x1 = clamp(min(xs) - 20, 0, w - 1)
        y1 = clamp(min(ys) - 20, 0, h - 1)
        x2 = clamp(max(xs) + 20, 0, w)
        y2 = clamp(max(ys) + 20, 0, h)

        fx, fy, fw, fh = x1, y1, x2 - x1, y2 - y1

        pitch, yaw, roll = estimate_head_pose(face_landmarks, w, h)
        if pitch is not None:
            pose_text = head_pose_label(pitch, yaw, roll)
            cv2.putText(frame, f'Head: {pose_text}', (25, 220),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 180, 80), 2)
            cv2.putText(frame, f'P:{pitch:.1f} Y:{yaw:.1f} R:{roll:.1f}', (25, 245),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 180, 80), 2)

        face_crop = frame[fy:fy+fh, fx:fx+fw]
        if face_crop.size > 0:
            face_rgb_crop = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)

            ml = face_landmarks[MOUTH_LEFT]
            mr = face_landmarks[MOUTH_RIGHT]
            mt = face_landmarks[MOUTH_TOP]
            mb = face_landmarks[MOUTH_BOTTOM]

            mx1 = clamp(int(min(ml.x, mr.x) * w) - 15, 0, w - 1)
            mx2 = clamp(int(max(ml.x, mr.x) * w) + 15, 0, w)
            my1 = clamp(int(min(mt.y, mb.y) * h) - 15, 0, h - 1)
            my2 = clamp(int(max(mt.y, mb.y) * h) + 25, 0, h)

            mouth_crop = frame[my1:my2, mx1:mx2]

            if mouth_crop.size > 0:
                mouth_rgb = cv2.cvtColor(mouth_crop, cv2.COLOR_BGR2RGB)
                mouth_pil = Image.fromarray(mouth_rgb).resize((224, 224))
                face_pil = Image.fromarray(face_rgb_crop).resize((224, 224))

                eyes_tensor = eyes_transform(face_pil).unsqueeze(0).to(device)
                yawn_tensor = yawn_transform(mouth_pil).unsqueeze(0).to(device)

                with torch.no_grad():
                    eyes_logits = eyes_model(eyes_tensor)
                    eyes_probs = torch.softmax(eyes_logits, 1)
                    eyes_pred = int(torch.argmax(eyes_probs, 1).item())
                    eyes_conf = float(eyes_probs.max().item())
                    raw_eye_closed = (eyes_pred == 0) and (eyes_conf > CONF_THRESHOLD)

                    if stable_eye_state == raw_eye_closed:
                        stable_eye_frames += 1
                    else:
                        stable_eye_state = raw_eye_closed
                        stable_eye_frames = 1

                    if stable_eye_frames >= STABLE_FRAMES:
                        cached_eyes_closed = is_closed_now = raw_eye_closed

                    yawn_logits = yawn_model(yawn_tensor)
                    yawn_probs = torch.softmax(yawn_logits, 1)
                    yawn_pred = int(torch.argmax(yawn_probs, 1).item())
                    yawn_conf = float(yawn_probs.max().item())
                    raw_yawn = (yawn_pred == 1) and (yawn_conf > YAWN_THRESHOLD)

                    if stable_yawn_state == raw_yawn:
                        stable_yawn_frames += 1
                    else:
                        stable_yawn_state = raw_yawn
                        stable_yawn_frames = 1

                    if stable_yawn_frames >= STABLE_FRAMES:
                        cached_is_yawn = is_yawn_now = raw_yawn

    if was_closed_prev and not is_closed_now:
        blink_count += 1
    if was_yawn_prev and not is_yawn_now:
        yawn_count += 1

    if is_closed_now:
        if eyes_closed_since is None:
            eyes_closed_since = now
    else:
        eyes_closed_since = None

    was_closed_prev = is_closed_now
    was_yawn_prev = is_yawn_now

    closed_duration = 0.0 if eyes_closed_since is None else now - eyes_closed_since

    if closed_duration >= CLOSED_ALARM_SECONDS:
        alarm_active = True
        flash = (0, 0, 255) if int(now * 5) % 2 else (255, 255, 255)
        cv2.rectangle(frame, (0, 0), (w, h), flash, 20)
        cv2.putText(frame, '🚨 WAKE UP! SLAPEN GEDETECTEERD 🚨', (160, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 5)
        if now - last_alarm >= 3.0:
            last_alarm = now
            start_loud_alarm()
    else:
        alarm_active = False

    if result.multi_face_landmarks:
        cv2.rectangle(frame, (fx, fy), (fx + fw, fy + fh), (255, 200, 0), 2)
        cv2.rectangle(frame, (mx1, my1), (mx2, my2), (0, 255, 0), 2)

        color = (0, 0, 255) if is_closed_now else (0, 255, 0)
        status = 'SLAPEN🔴' if is_closed_now else 'OK✅'
        label = f'{status} | E:{eyes_conf:.1f} Y:{yawn_conf:.1f}'
        cv2.putText(frame, label, (fx, max(fy - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    overlay = frame.copy()
    cv2.rectangle(overlay, (15, 15), (600, 285), (20, 20, 20), -1)
    frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)

    cv2.putText(frame, f'👁️  Blinks: {blink_count}', (25, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(frame, f'😴 Yawns:  {yawn_count}', (25, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 200, 0), 2)
    cv2.putText(frame, f'Eyes closed: {closed_duration:.1f}s', (25, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
    cv2.putText(frame, f'Yawn conf: {yawn_conf:.1f}', (25, 135), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
    cv2.putText(frame, f'FPS: {fps:.1f}', (25, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (150, 150, 150), 2)
    cv2.putText(frame, f'Head: {pose_text}', (25, 220), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 180, 80), 2)
    cv2.putText(frame, 'q=quit | MediaPipe = head pose only', (25, 255),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 2)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

    cv2.imshow('Fatigue Monitor (Head Pose + Old Models)', frame)

cap.release()
cv2.destroyAllWindows()
face_mesh.close()
print("✅ Done")

Imports geladen!
🚀 MPS!
✅ Eyes OK
✅ Yawn OK


I0000 00:00:1778163759.437874 1898111 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1778163759.441103 1900902 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778163759.446086 1900900 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


🎥 MEDIAPIPE + OLD MODELS | q=quit
✅ Done
